# Is your model's confidence trustworthy? Calibration metrics

_Metrics & Evaluation — measuring models, data & statistics_

**A model is calibrated if, among the cases it calls "70% likely", about 70% actually happen.**

Calibration is about honesty, not accuracy. A weather forecaster who says "70% chance of rain" is calibrated if, across all the days she said 70%, it actually rained on about 70% of them. She is right to be unsure — being calibrated does not mean being certain.

       So we group the model's predictions by the confidence it stated. Take every case where it said roughly "70% likely". Then look at reality: of those cases, what fraction actually turned out positive? If that fraction is also about 70%, the model is calibrated there. Do this for every confidence level and you get the whole picture.

---

This notebook is a practice scaffold for the **Is your model's confidence trustworthy? Calibration metrics** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — scikit-learn

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4,
                                       random_state=42, stratify=y)

# A shallow random forest: a good RANKER but a poor PROBABILITY estimator.
clf = RandomForestClassifier(n_estimators=50, max_depth=4,
                             random_state=0).fit(Xtr, ytr)
p = clf.predict_proba(Xte)[:, 1]          # predicted P(class = 1)

# --- reliability diagram: stated confidence vs observed frequency ---
frac_pos, mean_pred = calibration_curve(yte, p, n_bins=10, strategy="uniform")
# strategy="quantile" gives ADAPTIVE (equal-count) bins instead.

# --- from-scratch Expected / Maximum Calibration Error ---
def ece_mce(y_true, prob, n_bins=10):
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    N = len(prob); ece = 0.0; mce = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        in_bin = (prob > lo) & (prob <= hi) if i > 0 else (prob >= lo) & (prob <= hi)
        if in_bin.sum() == 0:
            continue
        conf = prob[in_bin].mean()          # average claimed probability
        acc = y_true[in_bin].mean()         # observed positive frequency
        gap = abs(acc - conf)
        ece += (in_bin.sum() / N) * gap     # size-weighted average gap
        mce = max(mce, gap)                 # worst single bin
    return ece, mce

ece, mce = ece_mce(yte, p)
print("AUC :", round(roc_auc_score(yte, p), 4))          # ranking quality
print("ECE :", round(ece, 4), " MCE:", round(mce, 4))    # calibration
print("Brier (proper score):", round(brier_score_loss(yte, p), 4))

# --- recalibration: fit ON HELD-OUT FOLDS, judge on the test set ---
for method in ["isotonic", "sigmoid"]:   # isotonic = monotone fit; sigmoid = Platt
    cal = CalibratedClassifierCV(
        RandomForestClassifier(n_estimators=50, max_depth=4, random_state=0),
        method=method, cv=5).fit(Xtr, ytr)
    pc = cal.predict_proba(Xte)[:, 1]
    e, _ = ece_mce(yte, pc)
    print(method, "-> ECE:", round(e, 4),
          " Brier:", round(brier_score_loss(yte, pc), 4),
          " AUC:", round(roc_auc_score(yte, pc), 4))  # AUC barely moves

## Visualize the data & results

_A shallow random forest on breast-cancer data ranks tumors almost perfectly (AUC 0.992). But when it says "80% malignant", is it right 80% of the time?_

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.metrics import brier_score_loss, roc_auc_score

X, y = load_breast_cancer(return_X_y=True)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4,
                                       random_state=42, stratify=y)

clf = RandomForestClassifier(n_estimators=50, max_depth=4,
                             random_state=0).fit(Xtr, ytr)
p = clf.predict_proba(Xte)[:, 1]

# reliability-diagram points (10 equal-width bins)
frac_pos, mean_pred = calibration_curve(yte, p, n_bins=10, strategy="uniform")
print("x (mean predicted):", np.round(mean_pred, 3))
print("y (obs frequency) :", np.round(frac_pos, 3))

def ece(y_true, prob, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1); N = len(prob); e = 0.0
    for i in range(n_bins):
        lo, hi = edges[i], edges[i + 1]
        m = (prob > lo) & (prob <= hi) if i > 0 else (prob >= lo) & (prob <= hi)
        if m.sum():
            e += (m.sum() / N) * abs(prob[m].mean() - y_true[m].mean())
    return e

print("raw  ECE:", round(ece(yte, p), 3), " AUC:", round(roc_auc_score(yte, p), 3))
for method in ["isotonic", "sigmoid"]:
    cal = CalibratedClassifierCV(
        RandomForestClassifier(n_estimators=50, max_depth=4, random_state=0),
        method=method, cv=5).fit(Xtr, ytr)
    pc = cal.predict_proba(Xte)[:, 1]
    print(method, "ECE:", round(ece(yte, pc), 3),
          " Brier:", round(brier_score_loss(yte, pc), 3),
          " AUC:", round(roc_auc_score(yte, pc), 3))

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A model's predictions, grouped into two confidence bins. Bin A: 100 cases, average predicted probability 0.30, 45 actually positive. Bin B: 100 cases, average predicted probability 0.90, 70 actually positive. Compute ECE and MCE.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Find each bin's observed frequency: $\mathrm{acc}(A)=45/100=0.45$, $\mathrm{acc}(B)=70/100=0.70$. — _The observed frequency is reality — the fraction of that bin that turned out positive._
- Find each bin's gap: $|0.45-0.30|=0.15$ for A, $|0.70-0.90|=0.20$ for B. — _The gap is how far the stated confidence is from what actually happened._
- Weight by bin size and average for ECE: each bin holds 100/200 = 0.5 of the cases, so $\mathrm{ECE}=0.5(0.15)+0.5(0.20)=0.175$. — _ECE weights each bin's gap by how many cases it contains, so crowded bins count more._
- Take the worst bin for MCE: $\mathrm{MCE}=\max(0.15,0.20)=0.20$. — _MCE reports the single largest miscalibration, here bin B at 0.20._

**Answer:** ECE = 0.175 and MCE = 0.20. Bin A is under-confident (says 0.30, reality 0.45); bin B is over-confident (says 0.90, reality 0.70).

</details>

**Problem 2.** A spam filter has AUC (Area Under the Curve) = 0.98 but its calibration curve sags far below the diagonal. A teammate says "AUC is high, so the 0.95 spam scores are trustworthy — auto-delete them." Why is that wrong, and what would you do?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Separate what each metric measures: AUC grades only ranking — whether spam outscores ham. It says nothing about whether "0.95" means a 95% chance. — _A model can rank perfectly yet attach systematically inflated probabilities._
- Read the reliability diagram: the curve sagging below $y=x$ means over-confidence — cases the model calls 0.95 are positive far less than 95% of the time. — _Below the diagonal, observed frequency is lower than the claimed probability._
- Recalibrate on held-out data with Platt scaling (sigmoid) or isotonic regression, then re-check ECE and the reliability curve on the test set. — _Recalibration bends the scores back toward the diagonal without disturbing the ranking, so AUC is preserved while the probabilities become honest._

**Answer:** High AUC only proves good ranking, not honest probabilities. The auto-delete threshold relies on the value 0.95 meaning 95%, which the sagging curve disproves. Recalibrate (Platt / isotonic) on a separate split, confirm ECE drops and the curve hugs $y=x$, then set the threshold from the calibrated scores.

</details>

**Problem 3.** A model predicts the base rate 0.60 for every single case. Is it calibrated? Is it useful? What metric exposes the problem?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check calibration: every prediction is 0.60, and across all those cases 60% are positive, so the gap is zero in the one occupied bin — it is perfectly calibrated. — _Calibration only asks that stated confidence match observed frequency, which this trivially satisfies._
- Check usefulness: it never distinguishes one case from another, so it cannot drive any decision. It has zero sharpness (no spread in its predictions). — _Sharpness measures how decisive predictions are; a constant output has none._
- Use the Brier decomposition: reliability is 0 (calibrated) but resolution is also 0 (the bins never differ from the base rate), so the Brier score is just the uncertainty $\bar{o}(1-\bar{o})=0.6\times0.4=0.24$. — _Resolution rewards separating outcomes; a constant model earns none, exposing the uselessness that calibration alone hides._

**Answer:** Yes, perfectly calibrated, but useless. Calibration alone cannot see this. The Brier score's resolution term (and sharpness) is 0, revealing that the model never commits — good probabilities must be calibrated and sharp.

</details>